In [1]:
import os
import sys
import json
from datetime import datetime

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

# Set environment variables before importing transformers
os.environ["HUGGINGFACE_HUB_CACHE"] = (
    "/scratch/shareddata/dldata/huggingface-hub-cache/hub"  # Load local models
)
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = (
    f"{os.environ["WRKDIR"]}/.cache/huggingface"  # Cache in work directory
)

from vllm import LLM, SamplingParams
import transformers
import numpy as np
from datasets import load_dataset

from utils.helpers import (
    get_system_prompt,
    get_default_response,
    make_prompt,
    parse_output,
    sample_dataset
)

# Force reload changes
from utils.prompts import *

from utils.constants import (
    PIPE_MAX_NEW_TOKENS,
    MODEL_TEMPERATURE,
    BATCH_SIZE,
    PIPE_RETURN_FULL_TEXT
)

INFO 04-01 20:20:59 [__init__.py:216] Automatically detected platform cuda.


In [2]:
LABELS = ["yes", "no"]  # Labels per column
EVAL_COLS = [
    "The exercise description matched the selected theme (Yes/No)",
    "The exercise description matched the selected topic (Yes/No)",
    "Included concepts that were too advanced (Yes/No)",
]
PRED_COLS = ["augmentedProblemDescription", "augmentedExampleSolution"]

DEFAULT_DATA = "./data/final_dataset.csv"
DEFAULT_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

In [3]:
task = "detect"
n_rows = 10
use_instructions = True
seed = 42
number_of_demonstrations = 0
type_of_demonstrations = 0

dataset = load_dataset("csv", data_files=DEFAULT_DATA, split="train", sep=";")
dataset = dataset.shuffle(seed=seed)

demonstrations = sample_dataset(
    dataset, seed, 0, type_of_demonstrations
)

system_prompt = get_system_prompt(task, demonstrations, bool(use_instructions))

dataset = dataset.map(
    lambda row: {
        "user_prompt": make_prompt(row, task),
        "system_prompt": system_prompt,
    },
)

if n_rows is not None and n_rows > 0:
    dataset = dataset.select(range(n_rows))

In [4]:
for row in dataset:
    #print(row["system_prompt"])
    break

In [5]:
model = DEFAULT_MODEL
#model = 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'

params = {
    "model": model,
    "device_map": 0,  # Force GPU
    "max_new_tokens": PIPE_MAX_NEW_TOKENS,
    "temperature": MODEL_TEMPERATURE
}

#pipeline = transformers.pipeline("text-generation", **params)
#pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
#pipeline.model.config.pad_token_id = pipeline.model.config.eos_token_id
mode = DEFAULT_MODEL.split("/")[0]

In [5]:
model = DEFAULT_MODEL


mode = "mistral" if "mistral" in model.lower() else "auto"
llm = LLM(model=model,
    gpu_memory_utilization=0.8,
    max_model_len=2048,
    max_num_seqs=16,
    enforce_eager=True,
    tokenizer_mode=mode,
    config_format=mode,
    load_format=mode
)

INFO 04-01 20:21:04 [arg_utils.py:504] HF_HUB_OFFLINE is True, replace model_id [meta-llama/Llama-3.1-8B-Instruct] to model_path [/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659]
INFO 04-01 20:21:04 [arg_utils.py:504] HF_HUB_OFFLINE is True, replace model_id [/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659] to model_path [/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659]
INFO 04-01 20:21:04 [utils.py:233] non-default args: {'max_model_len': 2048, 'gpu_memory_utilization': 0.8, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 04-01 20:21:17 [model.py:1682] Your device 'Tesla V100-PCIE-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 04-01 20:21:17 [model.py:1733] Casting torch.bfloat16 to torch.float16.
INFO 04-01 20:21:17 [model.py:1510] Using max model len 2048
INFO 04-01 20:21:20 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-01 20:21:20 [__init__.py:381] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=2043959) INFO 04-01 20:21:20 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=2043959) INFO 04-01 20:21:20 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--meta-llama--Llama-3.1-8B-Instruct/snapshots/0e9e39f249a16976918f6564b8830bc894c89659', speculative_config=None, tokenizer='/scratch/shareddata/dldata/huggingface-hub-cache/hub/models--meta-llama--Llama-3.1-8B-In

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]



KeyboardInterrupt



KeyboardInterrupt: 

In [ ]:
sampling_params = SamplingParams(temperature=0.3, max_tokens=250)

default_response = get_default_response(task)
results = {key: [] for key in default_response.keys()}

batch_size = BATCH_SIZE
prompts = dataset["user_prompt"]
system_prompts = dataset["system_prompt"]

complete_prompts = [
        [
            {"role": "system", "content": sp},
            {"role": "user", "content": up},
        ]
        for sp, up in zip(system_prompts, prompts)
]

output = llm.chat(complete_prompts, sampling_params, use_tqdm=True)

In [ ]:
for out in output:
    text = out.outputs[0].text
    parsed = parse_output(text)
    for key, value in default_response.items():
        results[key].append(json.dumps(parsed.get(key, value)))

for column_name, column_data in results.items():
    dataset = dataset.add_column(column_name, column_data)